In [13]:
# Load all artifacts needed for hyperparameter optimisation.
# setup_mlflow() must be called before any mlflow.start_run().

import sys
sys.path.insert(0, '..')

import time
import numpy as np
import pandas as pd
import joblib
import mlflow
import mlflow.sklearn
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.metrics import make_scorer

from src.estimators      import XGBoostDst, LightGBMDst
from src.mlflow_tracking import setup_mlflow
from src.runner   import fit_model, run_segment
from src.evaluate import compute_metrics

from sklearn.model_selection import RandomizedSearchCV



# Hyperparameter Optimisation

This notebook performs hyperparameter optimisation for XGBoost and LightGBM
at $h^* = 7$h with MODEL_C feature set (OMNI + $\delta n$, 21 features).

**Design:**
- Training: Train_1 + Train_2 (1995–2008)
- Optimisation: RandomizedSearchCV (n_iter=50, random_state=42) with TimeSeriesSplit (5 folds) on Train_1 + Train_2
- Val_Main and Val_Storm are **not** used during optimisation — they are independent external validation periods
- Scoring: negative Storm RMSE averaged across TimeSeriesSplit folds

**Depends on:**
- `data/processed/feat_split.parquet`
- `models/split_masks.pkl`
- `models/context_constants.pkl`
- `models/feature_selection_results.pkl`

**Produces:**
- `models/hp_opt_results.pkl` — best estimators, best params and full RandomizedSearchCV results for XGBoost and LightGBM

In [14]:
feat  = pd.read_parquet('../data/processed/feat_split.parquet')
masks = joblib.load('../models/split_masks.pkl')
ctx   = joblib.load('../models/context_constants.pkl')
fs    = joblib.load('../models/feature_selection_results.pkl')

mlflow.set_tracking_uri("../mlruns")
mlflow.set_experiment('cosmic_ray_storm_prediction')

<Experiment: artifact_location='file:///C:/Temp/python-start/work/PracticalProjects/cosmic-ray-storm/notebooks/../mlruns/292621262165687921', creation_time=1782395978156, effective_trace_archival_retention=None, experiment_id='292621262165687921', last_update_time=1782395978156, lifecycle_stage='active', name='cosmic_ray_storm_prediction', tags={}, trace_location=None, workspace='default'>

### Setup

Load all artifacts needed for hyperparameter optimisation. `setup_mlflow()` must be called before any `mlflow.start_run()`.

In [5]:
# ── Cell 1: Imports & Load ────────────────────────────────────────────────
#
# Load all artifacts needed for hyperparameter optimisation.
# setup_mlflow() must be called before any mlflow.start_run().

SELECTED_FEATURES = fs['selected_final']

NEUTRON_RAW     = ['neutron_counts']
NEUTRON_DERIVED = ['d_artefactsneutron']
NEUTRON_HISTORY = ['neutron_counts_lag3', 'neutron_counts_lag7']
NEUTRON_ALL     = NEUTRON_RAW + NEUTRON_DERIVED + NEUTRON_HISTORY

OMNI_FEATURES         = [f for f in SELECTED_FEATURES if f not in NEUTRON_ALL]
MODEL_C_OMNI_DNEUTRON = OMNI_FEATURES + NEUTRON_DERIVED

print(f'feat shape            : {feat.shape}')
print(f'MODEL_C features      : {len(MODEL_C_OMNI_DNEUTRON)}')
print(f'STORM_THR             : {STORM_THR} nT')

feat shape            : (364728, 72)
MODEL_C features      : 21
STORM_THR             : -50 nT


### Data Splits

Reconstruct `train1_mask` for MASE denominator. `X_train_full` and `y_train_full` use the full training set (Train_1 + Train_2) as the optimisation training set.

In [7]:
# ── Cell 2: Splits ────────────────────────────────────────────────────────
#
# Reconstruct Train_1 mask for MASE denominator.
# X_train_full and y_train_full use the full train set (Train_1 + Train_2)
# as the optimisation training set.
# ── Cell 2: Splits ────────────────────────────────────────────────────────
import pandas as pd

BOUNDARIES = ctx['BOUNDARIES']
PURGE_H    = ctx['PURGE_H']
dt         = feat['datetime']

def segment_mask(start_key, end_key, purge_start=True, purge_end=True):
    start = BOUNDARIES[start_key]
    end   = BOUNDARIES[end_key]
    if purge_start:
        start = start + pd.Timedelta(hours=PURGE_H)
    if purge_end:
        end   = end   - pd.Timedelta(hours=PURGE_H)
    return (dt >= start) & (dt <= end)

train_mask     = masks['train']       # Train_1 + Train_2
val_main_mask  = masks['val_main']
val_storm_mask = masks['val_storm']

# Full training set for optimisation
X_train_full = feat.loc[train_mask, MODEL_C_OMNI_DNEUTRON]
y_train_full = feat.loc[train_mask, 'dst_target_7h']

# y_train for MASE denominator (Train_1 only)
train1_mask = segment_mask('train1_start', 'train1_end',
                            purge_start=False, purge_end=True)
y_train     = feat.loc[train1_mask, 'dst'].copy()

print(f'Train_1+2 rows : {train_mask.sum():,}')
print(f'Val_main rows  : {val_main_mask.sum():,}')
print(f'Val_storm rows : {val_storm_mask.sum():,}')
print(f'NaN in y_train_full: {y_train_full.isna().sum()}')

Train_1+2 rows : 121,185
Val_main rows  : 52,542
Val_storm rows : 1,446
NaN in y_train_full: 0


### Custom Scorer & TimeSeriesSplit

`neg_storm_rmse` returns negative Storm RMSE — RandomizedSearchCV maximises score, so higher = lower Storm RMSE = better model. `TimeSeriesSplit` ensures chronological order is preserved across folds. Val_Main and Val_Storm are never used during optimisation.

In [8]:
# ── Cell 3: Custom scorer & TimeSeriesSplit ───────────────────────────────
#
# neg_storm_rmse: GridSearchCV maximises score — negative RMSE means
# higher score = lower Storm RMSE = better model.
# TimeSeriesSplit ensures chronological order is preserved across folds.
# Val_Main and Val_Storm are never used here.
# ── Cell 3: Custom scorer ─────────────────────────────────────────────────
#
# Negative Storm RMSE — averaged across TimeSeriesSplit folds.
# GridSearchCV maximises score, so negative RMSE = lower RMSE is better.
# Val_Main and Val_Storm are NOT used here — scoring is on CV folds only.

def neg_storm_rmse(y_true, y_pred, storm_thr=-50.0):
    """Negative Storm RMSE — higher is better for GridSearchCV."""
    mask = np.asarray(y_true) < storm_thr
    if mask.sum() == 0:
        return 0.0
    return -np.sqrt(np.mean(
        (np.asarray(y_true)[mask] - np.asarray(y_pred)[mask]) ** 2
    ))

storm_scorer = make_scorer(neg_storm_rmse, greater_is_better=True)
tscv         = TimeSeriesSplit(n_splits=5)

print('Scorer and TimeSeriesSplit ready.')

Scorer and TimeSeriesSplit ready.


### Reference Performance — Setup

Define evaluation segments. Both models are fitted with default hyperparameters on Train_1 + Train_2 to establish the pre-tuning baseline.

In [9]:
# ── Cell 4: Reference performance (before tuning) ─────────────────────────
#
# Establishes baseline against which tuned models are compared.
# Both models use default hyperparameters on Train_1 + Train_2.

EVAL_SEGMENTS = {
    'val_main' : val_main_mask,
    'val_storm': val_storm_mask,
}

### Reference Performance — XGBoost

Fit XGBoost with default hyperparameters. This model is used only as a reference baseline — the tuned model from RandomizedSearchCV replaces it for final evaluation.

In [10]:
# ── Cell 5: Fit XGBoost reference model ───────────────────────────────────
#
# Default hyperparameters. Used only as a reference baseline —
# the tuned model from RandomizedSearchCV replaces this for final evaluation.
a 
xgb_pipe  = Pipeline([('model', XGBoostDst())])
xgb_pipe.fit(X_train_full, y_train_full)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0",list,"['bz_gsm', 'sw...ed', 'sw...ty', 'sw...re', ...]"
,n_estimators,500
,max_depth,5
,learning_rate,0.05
,subsample,0.8
,colsample_bytree,0.8


### Reference Performance — LightGBM

Fit LightGBM with default hyperparameters. This model is used only as a reference baseline — the tuned model from RandomizedSearchCV replaces it for final evaluation.

In [11]:
# ── Cell 6: Fit LightGBM reference model ──────────────────────────────────
#
# Default hyperparameters. Used only as reference baseline —
# the tuned model from RandomizedSearchCV replaces this for final evaluation.

lgbm_pipe = Pipeline([('model', LightGBMDst())])
lgbm_pipe.fit(X_train_full, y_train_full)


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0",list,"['bz_gsm', 'sw...ed', 'sw...ty', 'sw...re', ...]"
,n_estimators,500
,max_depth,5
,learning_rate,0.05
,num_leaves,31
,subsample,0.8


### Reference Performance — Results

Evaluate both reference models on Val_Main and Val_Storm. These results are the baseline against which tuned models are compared. The better-performing reference model is not selected at this stage — model selection is performed only after hyperparameter optimisation.

In [15]:
# ── Cell 7: Print reference performance ───────────────────────────────────
#
# Evaluate both reference models on Val_Main and Val_Storm.
# These results are the baseline against which tuned models are compared.
print('\n── Reference performance (before tuning) ────────────────────────')
print(f'{"Model":<12} {"Segment":<12} {"RMSE":>8} {"StormRMSE":>12} {"MASE":>8}')
print('─' * 55)

for model_name, pipe in [('XGBoost', xgb_pipe), ('LightGBM', lgbm_pipe)]:
    for seg_name, seg_mask in EVAL_SEGMENTS.items():
        y_true    = feat.loc[seg_mask, 'dst_target_7h']
        y_pred    = pipe.predict(feat.loc[seg_mask, MODEL_C_OMNI_DNEUTRON])
        y_persist = feat.loc[seg_mask, 'dst'].values
        m = compute_metrics(
            y_true    = y_true,
            y_pred    = y_pred,
            y_train   = y_train,
            y_persist = y_persist,
            storm_thr = STORM_THR,
            horizon   = 7,
        )
        print(f'{model_name:<12} {seg_name:<12} '
              f'{m["rmse"]:>8.2f} {m["storm_rmse"]:>12.2f} {m["mase"]:>8.3f}')


── Reference performance (before tuning) ────────────────────────
Model        Segment          RMSE    StormRMSE     MASE
───────────────────────────────────────────────────────
XGBoost      val_main        14.48        27.29    3.435
XGBoost      val_storm       39.37       102.37    5.526
LightGBM     val_main        14.47        27.03    3.462
LightGBM     val_storm       40.01       104.25    5.608


### XGBoost RandomizedSearchCV

50 random combinations × 5 folds = 250 fits. Results logged to MLflow.

In [16]:
# ── XGBoost RandomizedSearchCV ────────────────────────────────────────────
#
# 50 random combinations × 5 folds = 250 fits
# Val_Main and Val_Storm are NOT used — scoring on CV folds only.

param_dist_xgb = {
    'model__n_estimators'    : [300, 500, 800, 1000],
    'model__max_depth'       : [3, 5, 7],
    'model__learning_rate'   : [0.005, 0.01, 0.05],
    'model__subsample'       : [0.7, 0.8, 1.0],
    'model__colsample_bytree': [0.7, 0.8, 1.0],
}

xgb_search = RandomizedSearchCV(
    estimator           = Pipeline([('model', XGBoostDst())]),
    param_distributions = param_dist_xgb,
    n_iter              = 50,
    cv                  = tscv,
    scoring             = storm_scorer,
    n_jobs              = 1,
    verbose             = 1,
    random_state        = 42,
    refit               = True,
)

t0 = time.time()
xgb_search.fit(X_train_full, y_train_full)
print(f'\nXGBoost time     : {(time.time()-t0)/60:.1f} min')
print(f'XGBoost params   : {xgb_search.best_params_}')
print(f'XGBoost CV score : {xgb_search.best_score_:.4f}')

Fitting 5 folds for each of 50 candidates, totalling 250 fits

XGBoost time     : 36.0 min
XGBoost params   : {'model__subsample': 0.8, 'model__n_estimators': 500, 'model__max_depth': 5, 'model__learning_rate': 0.005, 'model__colsample_bytree': 0.8}
XGBoost CV score : -34.8043


In [20]:
mlflow.end_run()

with mlflow.start_run(run_name='xgb_randomizedsearch'):
    mlflow.set_tag('model_type',  'xgboost')
    mlflow.set_tag('search_type', 'RandomizedSearchCV')
    mlflow.set_tag('n_iter', 50)
    mlflow.log_params(xgb_search.best_params_)
    mlflow.log_metric('cv_storm_rmse', -xgb_search.best_score_)
    mlflow.sklearn.log_model(
        xgb_search.best_estimator_,
        name='best_pipeline',
        skops_trusted_types=[
            'src.estimators.xgboost_dst.XGBoostDst',
            'xgboost.core.Booster',
            'xgboost.sklearn.XGBRegressor',
        ]
    )
    print('MLflow run logged.')

MLflow run logged.


### LightGBM RandomizedSearchCV

Full grid has 972 combinations — exhaustive search would take several hours. `RandomizedSearchCV` with `n_iter=50` and `random_state=42` samples 50 random combinations × 5 folds = 250 fits. Results logged to MLflow.

In [21]:
# ── LightGBM RandomizedSearchCV ──────────────────────────────────────────
#
# Val_Main and Val_Storm are NOT used — scoring on CV folds only.

param_dist_lgbm = {
    'model__n_estimators'    : [300, 500, 800, 1000],
    'model__max_depth'       : [3, 5, 7],
    'model__learning_rate'   : [0.005, 0.01, 0.05],
    'model__subsample'       : [0.7, 0.8, 1.0],
    'model__colsample_bytree': [0.7, 0.8, 1.0],
    'model__num_leaves'      : [15, 31, 63],
}

lgbm_search = RandomizedSearchCV(
    estimator           = Pipeline([('model', LightGBMDst())]),
    param_distributions = param_dist_lgbm,
    n_iter              = 50,
    cv                  = tscv,
    scoring             = storm_scorer,
    n_jobs              = 1,
    verbose             = 1,
    random_state        = 42,
    refit               = True,
)

t0 = time.time()
lgbm_search.fit(X_train_full, y_train_full)
lgbm_elapsed = (time.time() - t0) / 60

print(f'\nLightGBM RandomizedSearch time : {lgbm_elapsed:.1f} min')
print(f'LightGBM best params           : {lgbm_search.best_params_}')
print(f'LightGBM best CV score         : {lgbm_search.best_score_:.4f}')

Fitting 5 folds for each of 50 candidates, totalling 250 fits

LightGBM RandomizedSearch time : 25.3 min
LightGBM best params           : {'model__subsample': 1.0, 'model__num_leaves': 15, 'model__n_estimators': 500, 'model__max_depth': 3, 'model__learning_rate': 0.01, 'model__colsample_bytree': 1.0}
LightGBM best CV score         : -33.7432


In [22]:
# ── MLflow logging ────────────────────────────────────────────────────────
mlflow.end_run()

with mlflow.start_run(run_name='lgbm_randomizedsearch'):
    mlflow.set_tag('model_type',  'lightgbm')
    mlflow.set_tag('search_type', 'RandomizedSearchCV')
    mlflow.set_tag('n_iter', 50)
    mlflow.log_params(lgbm_search.best_params_)
    mlflow.log_metric('cv_storm_rmse', -lgbm_search.best_score_)
    mlflow.log_metric('search_time_min', lgbm_elapsed)
    mlflow.sklearn.log_model(
        lgbm_search.best_estimator_,
        name='best_pipeline',
        skops_trusted_types=[
            'src.estimators.lightgbm_dst.LightGBMDst',
            'lightgbm.basic.Booster',
            'lightgbm.sklearn.LGBMRegressor',
            'collections.OrderedDict',
        ]
    )
    print('MLflow run logged.')

MLflow run logged.


### Save Artifacts

Save best estimators and params for use in the main ML notebook. `hp_opt_results.pkl` contains full CV results for diagnostic plots.

In [29]:
# ── Save artifacts ────────────────────────────────────────────────────────
#
# Saves best estimators and params for use in the main ML notebook.
# hp_opt_results.pkl contains full CV results for diagnostic plots.

joblib.dump({
    'xgb_best_estimator' : xgb_search.best_estimator_,
    'xgb_best_params'    : xgb_search.best_params_,
    'xgb_cv_results'     : xgb_search.cv_results_,
    'lgbm_best_estimator': lgbm_search.best_estimator_,
    'lgbm_best_params'   : lgbm_search.best_params_,
    'lgbm_cv_results'    : lgbm_search.cv_results_,
}, '../models/hp_opt_results.pkl')

print('Saved: models/hp_opt_results.pkl')

Saved: models/hp_opt_results.pkl
